#Import

In [ ]:
# Imports
import glob
import os
import pandas as pd
import numpy as np
import regex as re
import html
import csv
import json
from datetime import datetime as dt

# Load Subreddits

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
subreddit_list = ["phinvest", "Philippines", "NintendoPH", "peyups",
"ADMU", "dlsu", "Tomasino", "filipinofood",
"OldSchoolPH", "PampamilyangPaoLUL", "3DSPH", "beautytalkph", "BPOinPH", "cagayandeoro", "Cebu",
"davao", "FilipinoFreethinkers", "FilmClubPH", "Iloilo",
"ilustrado", "indiemusicph", "KakaiBalita", "Kwaderno",
"LoLPHSubreddit", "mnl", "opm", "palawan",
"PBA", "PHBookClub",
"PHGamers", "PHikingAndBackpacking", "phlgbt",
"phr4r", "Pilipinas", "pinoyent",
"RedditPHCyclingClub", "Tagalog", "Tiangge", "Gulong",
"dostscholars", "AkoBaYungGago", 'Coronavirus_PH', 'exIglesiaNiCristo',
'studentsph', 'medschoolph', 'lawstudentsph', 'artph', 'FilipinoHistory',
'phclassifieds', 'Filipinology', 'CasualPH']

subreddit_list = sorted(subreddit_list)

In [ ]:
output_path = '/content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/RAW_COHFIE/online_forums/reddit'

# Preprocessing

In [ ]:
def preprocess_reddit(text):
    # html
    result = html.unescape(html.unescape(str(text)))

    # urls
    result = re.sub("\[(.+?)\]\(https?.*?\)(?=\s|$|[[:punct:]])", r"\1 XX_URL ", result)
    result = re.sub(r"\*\*(.+?)\*\*", r'\1', result) # bold
    result = re.sub(r"\*(.+?)\*", r'\1', result) # italicized

    result = re.sub(r"https?:\/*.*?(?=\s|$)", " XX_URL ", result)
    result = re.sub(r"^\[removed\]$", "", result)
    result = re.sub(r"^\[deleted\]$", "", result)
    # result = re.sub(r"^\(\s*removed\s*\)$", "", result)
    # result = re.sub(r"^\(\s*deleted\s*\)$", "", result)

    return result.strip()

def preprocess(text):
    # html
    result = html.unescape(html.unescape(str(text)))

    # Email
    result = re.sub(r"[\SÑñ]+@([\SÑñ]+\.)+[\SÑñ]+", " XX_EMAIL ", result)

    # urls
    result = re.sub("https?:\/\/([\w\-_]+\.)+([\w\-_]+)+(\/[^\s]+)*\/?", " XX_URL ", result, flags=re.IGNORECASE)
    result = re.sub(r"([\w\-]+\.)+(com|net|org|co|us|ph|io)(\/[^\s]+)*", " XX_URL ", result, flags=re.IGNORECASE)
    
    # twitter username mentions
    result = re.sub(r"(?<!\S)@[^\s.,!?]+(?!\S)", " XX_USERNAME ", result)

    # extra spaces
    result = re.sub(r'\s\s+',' ', result) # replace 2++ white space with 1 white space
    
    # source specific artifacts removal goes here
    result = re.sub(r"(\/)?r\/([^\s/]+)", "XX_SUBREDDIT", result)
    
    return result.strip()

# Separate by Date and apply Preprocessing

In [ ]:
path = '/content/drive/Shareddrives/DOST FilWordNet x WordSense/Thesis/Data Collection/Raw Data/Reddit/raw files'
filenames = [i for i in glob.glob(f'{path}/*.json')]

In [ ]:
combined_json = pd.concat([pd.read_json(f) for f in filenames])

combined_json

,author,created_utc,full_link,text,subreddit,title,created
0,kriiiiiish,2021-07-20 11:59:53,https://www.reddit.com/r/3DSPH/comments/onuflh...,,3DSPH,Police Chase Armed Robbers Who Almost Got Away...,1626753593
1,kenroubii,2021-06-25 16:26:46,https://www.reddit.com/r/3DSPH/comments/o7jio6...,For 3DS and other Nintendo topics and discussi...,3DSPH,Redirect to r/NintendoPH for 3DS and other Nin...,1624609606
2,oofboi3600,2020-12-11 08:06:00,https://www.reddit.com/r/3DSPH/comments/kar3eo...,magkano po sa greenhills mag repair ng new3ds ...,3DSPH,meron po ako na tanong,1607645160
3,StartFabulous9972,2020-12-08 23:26:11,https://www.reddit.com/r/3DSPH/comments/k96567...,,3DSPH,Masturbation With Corner,1607441171
4,hey-akm,2020-07-29 10:21:42,https://www.reddit.com/r/3DSPH/comments/hzsa7v...,San po kaya ako pwede magpa repair ng Touch sc...,3DSPH,3Ds XL touch screen needs repair,1595989302
...,...,...,...,...,...,...,...
7312,eyooji,2019-02-01 20:51:57,https://www.reddit.com/r/studentsph/comments/a...,"Nung simula ng school year maayos ayos pa ko, ...",studentsph,I finally feel productive again,1549025517
7313,friablesoul,2019-02-01 13:23:55,https://www.reddit.com/r/studentsph/comments/a...,I went to Technological University of the Phil...,studentsph,Applying to TUP,1548998635
7314,MatangLabo,2019-01-31 20:56:34,https://www.reddit.com/r/studentsph/comments/a...,,studentsph,BAKIT NAKAKASTRESS MAGING ESTUDYANTE SA PILIPINAS,1548939394
7315,LowMatt,2019-01-31 20:04:28,https://www.reddit.com/r/studentsph/comments/a...,i didn't do any computation for the math secti...,studentsph,UPCAT woes echo chamber thread,1548936268


In [ ]:
combined_json = combined_json.sort_values(by='created_utc')

combined_json

,author,created_utc,full_link,text,subreddit,title,created
433266,arnimation,2008-07-04 22:12:49,https://www.reddit.com/r/Philippines/comments/...,,Philippines,Blog Wars? chikatime.com | Blog.MonsterComments,1215180769
433265,[deleted],2009-03-18 23:55:25,https://www.reddit.com/r/Philippines/comments/...,,Philippines,Filipino Woman Who Accused U.S. Marine of Rape...,1237391725
433264,[deleted],2009-03-29 12:36:01,https://www.reddit.com/r/Philippines/comments/...,,Philippines,Government succumbs to Abu Sayyaf threats,1238301361
433263,[deleted],2009-03-30 04:57:38,https://www.reddit.com/r/Philippines/comments/...,,Philippines,No hostage freed despite military pullout,1238360258
433262,[deleted],2009-03-31 15:36:08,https://www.reddit.com/r/Philippines/comments/...,,Philippines,ABU SAYYAF GIVES OFFICIALS ULTIMATUM ON PULLOUT,1238484968
...,...,...,...,...,...,...,...
1,Agreeable_Border6405,2022-03-09 09:05:08,https://www.reddit.com/r/phr4r/comments/t9vu4u...,[removed],phr4r,[F4A] Glucose Parent,1646787908
0,Unlikely_Piece_5859,2022-03-09 09:08:36,https://www.reddit.com/r/dlsu/comments/t9vwh3/...,I'm willing pay CASH in exchange for a slot in...,dlsu,WILLING TO PAY: GEWORLD Y13 $$$$$$,1646788116
0,Lexiii009,2022-03-09 09:14:19,https://www.reddit.com/r/phinvest/comments/t9w...,"Are you Bitcoin? Because for you, I'm willing ...",phinvest,Reposting For thesis. Only few respondents lef...,1646788459
0,Lexiii009,2022-03-09 09:16:32,https://www.reddit.com/r/phclassifieds/comment...,"Are you Bitcoin? Because for you, I'm willing ...",phclassifieds,For thesis please help. Only few respondents left,1646788592


In [ ]:
# run preprocess on text
combined_json['text'] = combined_json['text'].apply(lambda text: preprocess(preprocess_reddit(text)))

In [ ]:
# combined_json[combined_json['processed_text'].apply(lambda text: " http " in text if text is not None else False)][['text', 'processed_text']].head(50).style

In [ ]:
# drop rows with null text
combined_json['text'].replace('', np.nan, inplace=True)
combined_json = combined_json.dropna(subset=['text'])
# combined_json = combined_json.drop(combined_json[combined_json['text'] == ''].index)

combined_json

,author,created_utc,full_link,text,subreddit,title,created
433243,sinus,2009-12-17 15:38:39,https://www.reddit.com/r/Philippines/comments/...,Anyone from the Philippines with a CS:Source s...,Philippines,Counter-strike: Source Server?,1261035519
433239,polite_tourette,2010-01-22 01:33:02,https://www.reddit.com/r/Philippines/comments/...,Sriracha has been my new favorite salty condim...,Philippines,Bagoong and.....,1264095182
433237,mentat,2010-02-10 13:49:39,https://www.reddit.com/r/Philippines/comments/...,Who are you voting for? Why? Let's get some re...,Philippines,So r/philippines/ what are your thoughts on th...,1265780979
433234,blue_horse_shoe,2010-02-24 10:11:47,https://www.reddit.com/r/Philippines/comments/...,"Somewhere close to transport, malls, airport, ...",Philippines,Where is the best place to stay in Manila,1266977507
433232,takemewithyou,2010-02-25 16:45:57,https://www.reddit.com/r/Philippines/comments/...,"I'm looking for decent, easily accessible musi...",Philippines,Recommendations for music bars within metro \r...,1267087557
...,...,...,...,...,...,...,...
3,Creativ3narrative,2022-03-09 09:04:44,https://www.reddit.com/r/phr4r/comments/t9vtvo...,Hello. Anyone looking for a nice company? I go...,phr4r,26 [M4F] SFW or NSFW Hangout - Quezon City,1646787884
2,gustokoramen,2022-03-09 09:04:55,https://www.reddit.com/r/phr4r/comments/t9vtzj...,Tanggal talaga angas ko pag may naging constan...,phr4r,21 [F4M] Patanggal angas pls,1646787895
0,Unlikely_Piece_5859,2022-03-09 09:08:36,https://www.reddit.com/r/dlsu/comments/t9vwh3/...,I'm willing pay CASH in exchange for a slot in...,dlsu,WILLING TO PAY: GEWORLD Y13 $$$$$$,1646788116
0,Lexiii009,2022-03-09 09:14:19,https://www.reddit.com/r/phinvest/comments/t9w...,"Are you Bitcoin? Because for you, I'm willing ...",phinvest,Reposting For thesis. Only few respondents lef...,1646788459


In [ ]:
# drop rows that has AutoModerator as author
combined_json.drop(combined_json[combined_json['author'] == 'AutoModerator'].index)

,author,created_utc,full_link,text,subreddit,title,created
433243,sinus,2009-12-17 15:38:39,https://www.reddit.com/r/Philippines/comments/...,Anyone from the Philippines with a CS:Source s...,Philippines,Counter-strike: Source Server?,1261035519
433239,polite_tourette,2010-01-22 01:33:02,https://www.reddit.com/r/Philippines/comments/...,Sriracha has been my new favorite salty condim...,Philippines,Bagoong and.....,1264095182
433237,mentat,2010-02-10 13:49:39,https://www.reddit.com/r/Philippines/comments/...,Who are you voting for? Why? Let's get some re...,Philippines,So r/philippines/ what are your thoughts on th...,1265780979
433234,blue_horse_shoe,2010-02-24 10:11:47,https://www.reddit.com/r/Philippines/comments/...,"Somewhere close to transport, malls, airport, ...",Philippines,Where is the best place to stay in Manila,1266977507
433232,takemewithyou,2010-02-25 16:45:57,https://www.reddit.com/r/Philippines/comments/...,"I'm looking for decent, easily accessible musi...",Philippines,Recommendations for music bars within metro \r...,1267087557
...,...,...,...,...,...,...,...
25,griftertm,2022-03-09 07:40:00,https://www.reddit.com/r/Philippines/comments/...,WASINGTON DC - Naghain ng guilty plea ang Pino...,Philippines,"Pinoy na binansagang 'Walis Tambo' man, naghai...",1646782800
22,captainbarbell,2022-03-09 07:51:35,https://www.reddit.com/r/Philippines/comments/...,"Considering they got ""60%"" in the surveys? I s...",Philippines,What are your opinions on why the UNITEAM rall...,1646783495
27,soju-soju,2022-03-09 07:57:59,https://www.reddit.com/r/phr4r/comments/t9uil2...,hello its me again. come join my nsfw channel ...,phr4r,23 [F4A] nsfw channel pt 2,1646783879
22,AshamedLoquat58,2022-03-09 08:13:24,https://www.reddit.com/r/phr4r/comments/t9utkl...,"i need someone to just drool over me, men and ...",phr4r,29 [M4A] Looking for someone who will fantasiz...,1646784804


In [ ]:
#sort by date
combined_json = combined_json.sort_values(by='created_utc')

combined_json

,author,created_utc,full_link,text,subreddit,title,created
433243,sinus,2009-12-17 15:38:39,https://www.reddit.com/r/Philippines/comments/...,Anyone from the Philippines with a CS:Source s...,Philippines,Counter-strike: Source Server?,1261035519
433239,polite_tourette,2010-01-22 01:33:02,https://www.reddit.com/r/Philippines/comments/...,Sriracha has been my new favorite salty condim...,Philippines,Bagoong and.....,1264095182
433237,mentat,2010-02-10 13:49:39,https://www.reddit.com/r/Philippines/comments/...,Who are you voting for? Why? Let's get some re...,Philippines,So r/philippines/ what are your thoughts on th...,1265780979
433234,blue_horse_shoe,2010-02-24 10:11:47,https://www.reddit.com/r/Philippines/comments/...,"Somewhere close to transport, malls, airport, ...",Philippines,Where is the best place to stay in Manila,1266977507
433232,takemewithyou,2010-02-25 16:45:57,https://www.reddit.com/r/Philippines/comments/...,"I'm looking for decent, easily accessible musi...",Philippines,Recommendations for music bars within metro \r...,1267087557
...,...,...,...,...,...,...,...
3,Creativ3narrative,2022-03-09 09:04:44,https://www.reddit.com/r/phr4r/comments/t9vtvo...,Hello. Anyone looking for a nice company? I go...,phr4r,26 [M4F] SFW or NSFW Hangout - Quezon City,1646787884
2,gustokoramen,2022-03-09 09:04:55,https://www.reddit.com/r/phr4r/comments/t9vtzj...,Tanggal talaga angas ko pag may naging constan...,phr4r,21 [F4M] Patanggal angas pls,1646787895
0,Unlikely_Piece_5859,2022-03-09 09:08:36,https://www.reddit.com/r/dlsu/comments/t9vwh3/...,I'm willing pay CASH in exchange for a slot in...,dlsu,WILLING TO PAY: GEWORLD Y13 $$$$$$,1646788116
0,Lexiii009,2022-03-09 09:14:19,https://www.reddit.com/r/phinvest/comments/t9w...,"Are you Bitcoin? Because for you, I'm willing ...",phinvest,Reposting For thesis. Only few respondents lef...,1646788459


In [ ]:
combined_json['subreddit'].value_counts()

phr4r                    229883
Philippines              116948
peyups                    25700
phinvest                  21762
phclassifieds             16446
CasualPH                  13244
dlsu                       9659
PHGamers                   6925
exIglesiaNiCristo          6262
studentsph                 6030
beautytalkph               5954
Tomasino                   5812
ADMU                       5611
RedditPHCyclingClub        4384
LawStudentsPH              4273
Cebu                       3332
Tagalog                    2471
davao                      2444
phlgbt                     1651
medschoolph                1616
NintendoPH                 1612
BPOinPH                    1159
cagayandeoro               1152
LoLPHSubreddit             1089
PHBookClub                 1003
PampamilyangPaoLUL          986
Iloilo                      939
dostscholars                828
Kwaderno                    724
FilmClubPH                  647
Coronavirus_PH              594
Gulong  

In [ ]:
#get year range
oldest_year = dt.strptime(combined_json['created_utc'].iloc[0], '%Y-%m-%d %H:%M:%S').year
earliest_year = dt.strptime(combined_json['created_utc'].iloc[len(combined_json) - 1], '%Y-%m-%d %H:%M:%S').year

In [ ]:
for year in range(oldest_year, earliest_year + 1):
  for month in range(1, 13):
    data = pd.DataFrame()

    for index, row in combined_json.iterrows():
      date = dt.strptime(row['created_utc'], '%Y-%m-%d %H:%M:%S').date()
      if date.year == year and date.month == month:
        data = data.append(row, ignore_index=True)

    json_path = f'{output_path}/{year}/{month}'

    if len(data.index) != 0: 
      if not os.path.exists(json_path):
        os.makedirs(json_path)
      data.to_json(f'{json_path}/{month}.json', orient = 'records', indent = 2)
      print(f'Data saved for {year}/{month}')

Data saved for 2009/12
Data saved for 2010/1
Data saved for 2010/2
Data saved for 2010/3
Data saved for 2010/4
Data saved for 2010/5
Data saved for 2010/8
Data saved for 2010/9
Data saved for 2010/10
Data saved for 2010/11
Data saved for 2010/12
Data saved for 2011/1
Data saved for 2011/2
Data saved for 2011/3
Data saved for 2011/4
Data saved for 2011/5
Data saved for 2011/6
Data saved for 2011/7
Data saved for 2011/8
Data saved for 2011/9
Data saved for 2011/10
Data saved for 2011/11
Data saved for 2011/12
Data saved for 2012/1
Data saved for 2012/2
Data saved for 2012/3
Data saved for 2012/4
Data saved for 2012/5
Data saved for 2012/6
Data saved for 2012/7
Data saved for 2012/8
Data saved for 2012/9
Data saved for 2012/10
Data saved for 2012/11
Data saved for 2012/12
Data saved for 2013/1
Data saved for 2014/1
Data saved for 2014/2
Data saved for 2014/3
Data saved for 2014/4
Data saved for 2014/5
Data saved for 2014/6
Data saved for 2014/7
Data saved for 2014/8
Data saved for 2014/9
